# 05 — Training-only feature-pipeline screening

This notebook screens feature-selection and feature-engineering choices using only the saved inner-training folds. It starts from the single final 26-feature table and retains two distinct feature-pipeline candidates for each split seed and target.

It does **not** inspect an outer test outcome, tune the nine model families, or estimate final performance. Notebook 06 will screen model families on the locally retained pipelines.


## 1. Setup

Load the numerical, statistical, and modeling tools used by the fold-safe screening workflow.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)


Locate the project and declare the final-run inputs and matching output directory.


In [2]:
working_directory = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [working_directory, *working_directory.parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "docs/research_protocol.md").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Launch this notebook from within the project directory.")

final_run_directory = project_root / "results/final_pipeline/06_final_fit_and_performance_summary"
aggregation_directory = final_run_directory / "02_patient_level_aggregation"
split_directory = final_run_directory / "03_setup_and_splits"
output_directory = final_run_directory / "05_feature_pipeline_screening"
output_directory.mkdir(parents=True, exist_ok=True)

input_paths = {
    "predictors": aggregation_directory / "primary_1040_26_predictors.csv",
    "outcomes": aggregation_directory / "primary_1040_outcome_metadata.csv",
    "feature_manifest": split_directory / "primary_feature_manifest.csv",
    "outer_splits": split_directory / "outer_split_assignments.csv",
    "inner_folds": split_directory / "inner_fold_assignments.csv",
}
missing_inputs = [name for name, path in input_paths.items() if not path.is_file()]
assert not missing_inputs, f"Missing required inputs: {missing_inputs}"

print("Project root:", project_root)
print("Output directory:", output_directory.relative_to(project_root))


Project root: /Users/rafsan_temp/Library/CloudStorage/OneDrive-SeattleUniversity/SU Projects/pd-fall-risk
Output directory: results/final_pipeline/06_final_fit_and_performance_summary/05_feature_pipeline_screening


## 2. Load and verify the locked analysis input

Read the unimputed 1,040-patient table, outcomes, feature order, and paired split assignments. Patient identifiers are kept as strings.


In [3]:
predictors = pd.read_csv(
    input_paths["predictors"],
    dtype={"PATNO": "string"},
    low_memory=False,
).set_index("PATNO")
outcome_metadata = pd.read_csv(
    input_paths["outcomes"],
    dtype={"PATNO": "string"},
    usecols=["PATNO", "falls_class"],
)
feature_manifest = pd.read_csv(input_paths["feature_manifest"])
outer_assignments = pd.read_csv(
    input_paths["outer_splits"],
    dtype={"PATNO": "string"},
)
inner_assignments = pd.read_csv(
    input_paths["inner_folds"],
    dtype={"PATNO": "string"},
)

features = feature_manifest["feature"].tolist()
outcomes = outcome_metadata.set_index("PATNO")["falls_class"].astype(int)

assert predictors.columns.tolist() == features
assert len(features) == 26
assert predictors.index.is_unique and outcomes.index.is_unique
assert set(predictors.index) == set(outcomes.index)
assert set(outer_assignments["PATNO"]) == set(predictors.index)
assert set(inner_assignments["PATNO"]).issubset(set(predictors.index))

print("Patients:", len(predictors))
print("Candidate source features:", len(features))
print("Outer split seeds:", outer_assignments["split_seed"].nunique())
print(
    "Inner-training partitions:",
    inner_assignments[["split_seed", "inner_validation_fold"]]
    .drop_duplicates()
    .shape[0],
)


Patients: 1040
Candidate source features: 26
Outer split seeds: 20
Inner-training partitions: 100


Confirm that every saved inner assignment contains exactly the corresponding outer-training patients and excludes that seed's test patients.


In [4]:
split_checks = []
for split_seed in sorted(outer_assignments["split_seed"].unique()):
    outer_train = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )
    inner_patients = set(
        inner_assignments.loc[
            inner_assignments["split_seed"].eq(split_seed),
            "PATNO",
        ]
    )
    split_checks.append({
        "split_seed": split_seed,
        "inner_equals_outer_train": inner_patients == outer_train,
        "outer_test_excluded": outer_test.isdisjoint(inner_patients),
    })

split_audit = pd.DataFrame(split_checks)
assert split_audit[["inner_equals_outer_train", "outer_test_excluded"]].all().all()
print(f"Split boundary checks passed: {len(split_audit)}/{len(split_audit)}")


Split boundary checks passed: 20/20


## 3. Clinical preprocessing rules

Declare nominal variables and the three missing-state representation groups used by the approved imputation strategy.


In [5]:
NOMINAL_FEATURES = {
    "DXPOSINS", "DXRIGID", "DOPTHERST", "FEATPOSHYP",
    "ANYFAMPD", "DXTREMOR", "DXBRADY", "DOMSIDE",
}
ORDINAL_FEATURES = {
    "FRZGT12M", "SCAU14", "SCAU16", "NP1SLPD", "NP1URIN",
    "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX", "NHY_COMBINED_MAX",
    "NP1CNST",
}
MISSING_INDICATORS = [
    "FOG_FORM_MISSING", "NQ_FORM_MISSING", "PART_IV_FORM_MISSING",
]

assert NOMINAL_FEATURES.issubset(features)
assert ORDINAL_FEATURES.issubset(features)
assert "NQ_GAUSSIAN_REVISION" in features


def representation_group(source):
    if source in {"FRZGT12M", "FOG_FORM_MISSING"}:
        return "GROUP_FREEZING_FORM"
    if source in {"NQ_GAUSSIAN_REVISION", "NQ_FORM_MISSING"}:
        return "GROUP_NEUROQOL_FORM"
    if source in {"NP4TOT", "PART_IV_FORM_MISSING"}:
        return "GROUP_PART_IV_FORM"
    return source


print("Nominal source features:", len(NOMINAL_FEATURES))
print("Ordinal source features:", len(ORDINAL_FEATURES))
print("Derived missing-state indicators:", len(MISSING_INDICATORS))


Nominal source features: 8
Ordinal source features: 9
Derived missing-state indicators: 3


Derive missing-state indicators before filling values and apply the fixed Part IV structural-zero rule only when therapy is recorded as `No`.


In [6]:
def clinical_raw(patient_ids):
    raw = predictors.loc[list(patient_ids), features].copy()

    indicators = pd.DataFrame(index=raw.index)
    indicators["FOG_FORM_MISSING"] = raw["FRZGT12M"].isna().astype(int)
    indicators["NQ_FORM_MISSING"] = raw["NQ_GAUSSIAN_REVISION"].isna().astype(int)
    indicators["PART_IV_FORM_MISSING"] = raw["NP4TOT"].isna().astype(int)

    structural_zero = raw["NP4TOT"].isna() & raw["DOPTHERST"].eq("No")
    raw.loc[structural_zero, "NP4TOT"] = 0.0

    return pd.concat([raw, indicators], axis=1)


Fit numeric fills, nominal categories, one-hot encoding, and scaling on one training fold, then apply those fitted transformations unchanged to validation patients.


In [7]:
def dense_pair(training_ids, validation_ids):
    raw_train = clinical_raw(training_ids)
    raw_validation = clinical_raw(validation_ids)

    nominal = [column for column in raw_train.columns if column in NOMINAL_FEATURES]
    numeric = [column for column in raw_train.columns if column not in nominal]
    train = raw_train.copy()
    validation = raw_validation.copy()

    freezing_mode = train["FRZGT12M"].mode(dropna=True)
    if freezing_mode.empty:
        raise ValueError("FRZGT12M has no observed value in this training fold.")
    train["FRZGT12M"] = train["FRZGT12M"].fillna(freezing_mode.iloc[0])
    validation["FRZGT12M"] = validation["FRZGT12M"].fillna(freezing_mode.iloc[0])

    median_columns = [column for column in numeric if column != "FRZGT12M"]
    medians = train[median_columns].median()
    if medians.isna().any():
        missing = medians[medians.isna()].index.tolist()
        raise ValueError(f"No training value available for: {missing}")
    train[median_columns] = train[median_columns].fillna(medians)
    validation[median_columns] = validation[median_columns].fillna(medians)

    for column in nominal:
        train[column] = train[column].astype("string").fillna("Missing").astype(str)
        validation[column] = (
            validation[column].astype("string").fillna("Missing").astype(str)
        )

    transformer = ColumnTransformer(
        [
            ("numeric", StandardScaler(), numeric),
            (
                "nominal",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                nominal,
            ),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    X_train = transformer.fit_transform(train)
    X_validation = transformer.transform(validation)

    source_by_column = {column: column for column in numeric}
    encoder = transformer.named_transformers_["nominal"]
    encoded_names = encoder.get_feature_names_out(nominal)
    encoded_sources = [
        source
        for source, categories in zip(nominal, encoder.categories_)
        for _ in categories
    ]
    source_by_column.update(dict(zip(encoded_names, encoded_sources)))
    group_by_column = {
        column: representation_group(source_by_column[column])
        for column in X_train.columns
    }

    assert X_train.notna().all().all()
    assert X_validation.notna().all().all()
    assert X_train.columns.equals(X_validation.columns)

    return raw_train, X_train, X_validation, group_by_column


## 4. Targets and performance measures

Construct the direct, Stage 1, and Stage 2 targets independently. Stage 2 training and validation contain only true fallers.


In [8]:
def target_data(target_name, training_ids, validation_ids):
    y_train = outcomes.loc[list(training_ids)]
    y_validation = outcomes.loc[list(validation_ids)]

    if target_name == "direct":
        return list(training_ids), list(validation_ids), y_train, y_validation
    if target_name == "stage_1":
        return (
            list(training_ids),
            list(validation_ids),
            y_train.gt(0).astype(int),
            y_validation.gt(0).astype(int),
        )

    y_train = y_train[y_train.gt(0)]
    y_validation = y_validation[y_validation.gt(0)]
    return (
        y_train.index.tolist(),
        y_validation.index.tolist(),
        y_train.eq(2).astype(int),
        y_validation.eq(2).astype(int),
    )


def classification_metrics(target_name, truth, predictions):
    labels = [0, 1, 2] if target_name == "direct" else [0, 1]
    recalls = recall_score(
        truth,
        predictions,
        labels=labels,
        average=None,
        zero_division=0,
    )
    result = {
        "macro_f1": f1_score(truth, predictions, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(truth, predictions),
        "accuracy": accuracy_score(truth, predictions),
        "recall_class_0": recalls[0],
        "recall_class_1": recalls[1],
        "recall_class_2": recalls[2] if target_name == "direct" else np.nan,
    }
    result["priority_recall"] = (
        result["recall_class_0"]
        if target_name == "stage_2"
        else result["recall_class_1"]
    )
    return result


## 5. Corrected fold-specific statistical selector

Calculate raw p-values using only the current inner-training patients, then apply Benjamini–Hochberg correction across that fold's candidate representations.


In [9]:
def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.flatnonzero(np.isfinite(p_values))
    if not len(valid):
        return adjusted

    order = valid[np.argsort(p_values[valid])]
    ranked = p_values[order] * len(valid) / np.arange(1, len(valid) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def numeric_p_value(values, target, ordinal):
    observed = pd.DataFrame({"value": values, "target": target}).dropna()
    groups = [
        observed.loc[observed["target"].eq(label), "value"].astype(float).to_numpy()
        for label in sorted(observed["target"].unique())
    ]
    if len(groups) < 2 or min(len(group) for group in groups) < 2:
        return np.nan
    if np.unique(np.concatenate(groups)).size < 2:
        return 1.0

    if ordinal:
        test = (
            stats.mannwhitneyu(*groups, alternative="two-sided")
            if len(groups) == 2
            else stats.kruskal(*groups)
        )
        return float(test.pvalue)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        normality = [
            stats.normaltest(group).pvalue
            if len(group) >= 8 and np.unique(group).size >= 3
            else 0.0
            for group in groups
        ]
        variance_p = stats.levene(*groups, center="median").pvalue

    parametric = (
        all(np.isfinite(normality))
        and min(normality) >= 0.05
        and np.isfinite(variance_p)
        and variance_p >= 0.05
    )
    if len(groups) == 2:
        test = (
            stats.ttest_ind(*groups, equal_var=True)
            if parametric
            else stats.mannwhitneyu(*groups, alternative="two-sided")
        )
    else:
        test = stats.f_oneway(*groups) if parametric else stats.kruskal(*groups)
    return float(test.pvalue)


def categorical_p_value(values, target, random_seed):
    categories = values.astype("string").fillna("Missing")
    table = pd.crosstab(categories, target)
    table = table.loc[table.sum(axis=1).gt(0)]
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 1.0

    asymptotic = stats.chi2_contingency(table, correction=False)
    sparse = (
        asymptotic.expected_freq.min() < 1
        or (asymptotic.expected_freq < 5).mean() > 0.20
    )
    if not sparse:
        return float(asymptotic.pvalue)

    method = stats.PermutationMethod(
        n_resamples=999,
        rng=np.random.default_rng(random_seed),
    )
    result = stats.chi2_contingency(
        table,
        correction=False,
        method=method,
    )
    return float(result.pvalue)


Implement group-aware FDR, L1, and Extra-Trees selectors. The L1 branch uses deterministic one-vs-rest `liblinear` fits so the direct three-class target is handled without SAGA convergence noise. If any member of a declared missing-state family is selected, the entire family is retained.


In [10]:
def fit_checked(estimator, X, y, context):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("error", category=ConvergenceWarning)
            estimator.fit(X, y)
    except ConvergenceWarning as error:
        raise RuntimeError(
            f"{context} did not converge. Increase max_iter before continuing."
        ) from error
    return estimator


def columns_for_groups(selected_groups, group_by_column):
    columns = [
        column
        for column, group in group_by_column.items()
        if group in selected_groups
    ]
    if not columns:
        raise RuntimeError("The selector returned no encoded columns.")
    return columns


def fdr_select(raw_train, target, group_by_column, random_seed):
    p_values = []
    for position, column in enumerate(raw_train.columns):
        if column in NOMINAL_FEATURES or column in MISSING_INDICATORS:
            p_value = categorical_p_value(
                raw_train[column],
                target,
                random_seed + position,
            )
        else:
            p_value = numeric_p_value(
                raw_train[column],
                target,
                ordinal=column in ORDINAL_FEATURES,
            )
        p_values.append(p_value)

    q_values = bh_adjust(p_values)
    selected_sources = [
        column
        for column, q_value in zip(raw_train.columns, q_values)
        if np.isfinite(q_value) and q_value <= 0.05
    ]
    if not selected_sources:
        finite = np.flatnonzero(np.isfinite(q_values))
        fallback = finite[np.argmin(q_values[finite])] if len(finite) else 0
        selected_sources = [raw_train.columns[fallback]]

    selected_groups = {
        representation_group(source)
        for source in selected_sources
    }
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def l1_select(X_train, target, group_by_column, C_value, random_seed):
    base_estimator = LogisticRegression(
        solver="liblinear",
        l1_ratio=1.0,
        C=C_value,
        class_weight="balanced",
        max_iter=5000,
        random_state=random_seed,
    )
    selector = OneVsRestClassifier(base_estimator, n_jobs=-1)
    fit_checked(selector, X_train, target, "L1 selector")
    coefficients = np.max(
        np.vstack([
            np.abs(estimator.coef_).reshape(-1)
            for estimator in selector.estimators_
        ]),
        axis=0,
    )
    selected_groups = {
        group_by_column[column]
        for column, coefficient in zip(X_train.columns, coefficients)
        if coefficient > 1e-10
    }
    if not selected_groups:
        strongest = X_train.columns[int(np.argmax(coefficients))]
        selected_groups = {group_by_column[strongest]}
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def tree_select(
    X_train,
    target,
    group_by_column,
    threshold_multiplier,
    random_seed,
):
    selector = ExtraTreesClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=random_seed,
        n_jobs=-1,
    )
    selector.fit(X_train, target)

    group_importance = {}
    for column, importance in zip(X_train.columns, selector.feature_importances_):
        group = group_by_column[column]
        group_importance[group] = group_importance.get(group, 0.0) + float(importance)

    threshold = np.median(list(group_importance.values())) * threshold_multiplier
    selected_groups = {
        group
        for group, importance in group_importance.items()
        if importance >= threshold
    }
    if not selected_groups:
        selected_groups = {max(group_importance, key=group_importance.get)}
    return columns_for_groups(selected_groups, group_by_column), selected_groups


def selected_feature_columns(
    selector_name,
    selector_parameter,
    raw_train,
    X_train,
    target,
    group_by_column,
    random_seed,
):
    if selector_name == "none":
        return X_train.columns.tolist(), set(group_by_column.values())
    if selector_name == "corrected_fdr":
        return fdr_select(raw_train, target, group_by_column, random_seed)
    if selector_name == "l1":
        C_value = float(selector_parameter.split("=")[1])
        return l1_select(
            X_train,
            target,
            group_by_column,
            C_value,
            random_seed,
        )

    multiplier = 1.25 if "1.25" in selector_parameter else 1.0
    return tree_select(
        X_train,
        target,
        group_by_column,
        multiplier,
        random_seed,
    )


## 6. Selector-screen configurations

Use the same fixed balanced logistic and shallow Extra-Trees references as the prior verified screening design. Duplicate reference models share one pipeline key so Notebook 06 receives distinct feature pipelines.


In [11]:
selector_configurations = [
    ("none", "all", "logistic"),
    ("corrected_fdr", "q<=0.05", "logistic"),
    ("l1", "C=0.1", "logistic"),
    ("l1", "C=1.0", "logistic"),
    ("none", "all", "extra_trees"),
    ("corrected_fdr", "q<=0.05", "extra_trees"),
    ("extra_trees", "threshold=median", "extra_trees"),
    ("extra_trees", "threshold=1.25*median", "extra_trees"),
]
selector_manifest = pd.DataFrame(
    selector_configurations,
    columns=["selector", "selector_parameter", "reference_model"],
)
selector_manifest.insert(
    0,
    "configuration_id",
    [f"SEL_{number:02d}" for number in range(1, len(selector_manifest) + 1)],
)
selector_manifest["candidate_kind"] = "selector_configuration"
selector_manifest["feature_set"] = "final_26"
selector_manifest["selector_implementation"] = np.select(
    [
        selector_manifest["selector"].eq("l1"),
        selector_manifest["selector"].eq("extra_trees"),
        selector_manifest["selector"].eq("corrected_fdr"),
    ],
    [
        "one_vs_rest_liblinear_l1",
        "shallow_extra_trees_group_importance",
        "within_fold_bh_fdr",
    ],
    default="no_supervised_removal",
)
selector_manifest["pipeline_key"] = (
    "raw__"
    + selector_manifest["selector"]
    + "__"
    + selector_manifest["selector_parameter"]
)

def reference_model(model_name, random_seed):
    if model_name == "logistic":
        return LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=random_seed,
        )
    return ExtraTreesClassifier(
        n_estimators=120,
        max_depth=6,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=random_seed,
        n_jobs=-1,
    )


display(selector_manifest)


,configuration_id,selector,selector_parameter,reference_model,candidate_kind,feature_set,selector_implementation,pipeline_key
0,SEL_01,none,all,logistic,selector_configuration,final_26,no_supervised_removal,raw__none__all
1,SEL_02,corrected_fdr,q<=0.05,logistic,selector_configuration,final_26,within_fold_bh_fdr,raw__corrected_fdr__q<=0.05
2,SEL_03,l1,C=0.1,logistic,selector_configuration,final_26,one_vs_rest_liblinear_l1,raw__l1__C=0.1
3,SEL_04,l1,C=1.0,logistic,selector_configuration,final_26,one_vs_rest_liblinear_l1,raw__l1__C=1.0
4,SEL_05,none,all,extra_trees,selector_configuration,final_26,no_supervised_removal,raw__none__all
5,SEL_06,corrected_fdr,q<=0.05,extra_trees,selector_configuration,final_26,within_fold_bh_fdr,raw__corrected_fdr__q<=0.05
6,SEL_07,extra_trees,threshold=median,extra_trees,selector_configuration,final_26,shallow_extra_trees_group_importance,raw__extra_trees__threshold=median
7,SEL_08,extra_trees,threshold=1.25*median,extra_trees,selector_configuration,final_26,shallow_extra_trees_group_importance,raw__extra_trees__threshold=1.25*median


## 7. Compatible feature-engineering branches

Define four interactions and one PCA group that can be constructed entirely from the final 26 source variables. PCA thresholds remain fixed at 0.80, 0.90, and 0.95 explained variance. The Neuro-QoL PCA branches are no longer possible because the Neuro-QoL questionnaire enters only as the single Gaussian score (decision D28).


In [12]:
interaction_sources = {
    "INT_GAIT_X_POSTURAL_STABILITY": [
        "NP3GAIT_COMBINED_MAX", "NP3PSTBL_COMBINED_MAX",
    ],
    "INT_MOTOR_X_COGNITION": ["NP3TOT_COMBINED_MAX", "MCATOT"],
    "INT_DURATION_X_MOTOR": [
        "Years_since_PD_diagnosis", "NP3TOT_COMBINED_MAX",
    ],
    "INT_MOBILITY_X_POSTURAL_STABILITY": [
        "NQ_GAUSSIAN_REVISION", "NP3PSTBL_COMBINED_MAX",
    ],
}
pca_groups = {
    "PART_I_PATIENT_3": ["NP1SLPD", "NP1URIN", "NP1CNST"],
}

engineering_rows = []
for branch, sources in interaction_sources.items():
    engineering_rows.append({
        "branch": branch,
        "branch_kind": "interaction",
        "source_features": " | ".join(sources),
        "variance_threshold": np.nan,
        "reference_model": "extra_trees",
        "pipeline_key": f"engineering__{branch}",
    })
for group_name, sources in pca_groups.items():
    for threshold in [0.80, 0.90, 0.95]:
        branch = f"PCA_{group_name}_{int(threshold * 100)}"
        engineering_rows.append({
            "branch": branch,
            "branch_kind": "pca",
            "source_features": " | ".join(sources),
            "variance_threshold": threshold,
            "reference_model": "extra_trees",
            "pipeline_key": f"engineering__{branch}",
        })

engineering_manifest = pd.DataFrame(engineering_rows)
engineering_manifest.insert(
    0,
    "configuration_id",
    [f"ENG_{number:02d}" for number in range(1, len(engineering_manifest) + 1)],
)
engineering_manifest["candidate_kind"] = "engineered_representation"
engineering_manifest["feature_set"] = "final_26"
engineering_manifest["selector"] = "none"
engineering_manifest["selector_parameter"] = "all"

assert len(interaction_sources) == 4
assert len(pca_groups) == 1
assert len(engineering_manifest) == 7
assert all(
    set(source_text.split(" | ")).issubset(features)
    for source_text in engineering_manifest["source_features"]
)

display(
    engineering_manifest[
        ["configuration_id", "branch", "branch_kind", "source_features", "variance_threshold"]
    ]
)


,configuration_id,branch,branch_kind,source_features,variance_threshold
0,ENG_01,INT_GAIT_X_POSTURAL_STABILITY,interaction,NP3GAIT_COMBINED_MAX | NP3PSTBL_COMBINED_MAX,NaN
1,ENG_02,INT_MOTOR_X_COGNITION,interaction,NP3TOT_COMBINED_MAX | MCATOT,NaN
2,ENG_03,INT_DURATION_X_MOTOR,interaction,Years_since_PD_diagnosis | NP3TOT_COMBINED_MAX,NaN
3,ENG_04,INT_MOBILITY_X_POSTURAL_STABILITY,interaction,NQ_GAUSSIAN_REVISION | NP3PSTBL_COMBINED_MAX,NaN
4,ENG_05,PCA_PART_I_PATIENT_3_80,pca,NP1SLPD | NP1URIN | NP1CNST,0.80
5,ENG_06,PCA_PART_I_PATIENT_3_90,pca,NP1SLPD | NP1URIN | NP1CNST,0.90
6,ENG_07,PCA_PART_I_PATIENT_3_95,pca,NP1SLPD | NP1URIN | NP1CNST,0.95


Record why the earlier larger engineering branches are not eligible for this 26-source final run. This is a design audit rather than a performance result.


In [13]:
excluded_legacy_branches = pd.DataFrame([
    {
        "earlier_branch": "INT_SBP_CHANGE_X_LIGHTHEADEDNESS",
        "status": "excluded",
        "reason": "SYS_CHANGE_STANDING_MINUS_SUPINE is outside the final 26-source universe",
        "final_26_replacement": "none",
    },
    {
        "earlier_branch": "PCA_neuroqol_items_8",
        "status": "excluded",
        "reason": "Individual Neuro-QoL items are not candidates; the questionnaire enters only as the single Gaussian score (D28)",
        "final_26_replacement": "none; NQ_GAUSSIAN_REVISION remains available as a source feature",
    },
    {
        "earlier_branch": "PCA_part_i_rater_items_6",
        "status": "excluded",
        "reason": "The six individual rater items are outside the final 26-source universe",
        "final_26_replacement": "none; NP1RTOT remains available as a source feature",
    },
    {
        "earlier_branch": "PCA_part_i_patient_items_7",
        "status": "redefined within final inputs",
        "reason": "Only sleepiness, urinary problems, and constipation are in the final 26",
        "final_26_replacement": "PCA_PART_I_PATIENT_3 at 0.80, 0.90, and 0.95",
    },
    {
        "earlier_branch": "PCA_part_iv_items_6",
        "status": "excluded",
        "reason": "The six individual Part IV items are outside the final 26-source universe",
        "final_26_replacement": "none; NP4TOT remains available as a source feature",
    },
    {
        "earlier_branch": "PCA_orthostatic_raw_vitals_6",
        "status": "excluded",
        "reason": "The raw orthostatic vital signs are outside the final 26-source universe",
        "final_26_replacement": "none",
    },
])

display(excluded_legacy_branches)


,earlier_branch,status,reason,final_26_replacement
0,INT_SBP_CHANGE_X_LIGHTHEADEDNESS,excluded,SYS_CHANGE_STANDING_MINUS_SUPINE is outside th...,none
1,PCA_neuroqol_items_8,excluded,Individual Neuro-QoL items are not candidates;...,none; NQ_GAUSSIAN_REVISION remains available a...
2,PCA_part_i_rater_items_6,excluded,The six individual rater items are outside the...,none; NP1RTOT remains available as a source fe...
3,PCA_part_i_patient_items_7,redefined within final inputs,"Only sleepiness, urinary problems, and constip...","PCA_PART_I_PATIENT_3 at 0.80, 0.90, and 0.95"
4,PCA_part_iv_items_6,excluded,The six individual Part IV items are outside t...,none; NP4TOT remains available as a source fea...
5,PCA_orthostatic_raw_vitals_6,excluded,The raw orthostatic vital signs are outside th...,none


Apply one interaction or one PCA representation after fold-specific imputation and scaling. Main effects remain with interactions; PCA replaces only its declared source columns.


In [14]:
def engineered_pair(X_train, X_validation, branch_row):
    train = X_train.copy()
    validation = X_validation.copy()
    sources = branch_row["source_features"].split(" | ")

    if branch_row["branch_kind"] == "interaction":
        branch = branch_row["branch"]
        train[branch] = train[sources[0]].to_numpy() * train[sources[1]].to_numpy()
        validation[branch] = (
            validation[sources[0]].to_numpy()
            * validation[sources[1]].to_numpy()
        )
        return train, validation

    pca = PCA(
        n_components=float(branch_row["variance_threshold"]),
        svd_solver="full",
    )
    training_components = pca.fit_transform(train[sources])
    validation_components = pca.transform(validation[sources])
    component_names = [
        f"{branch_row['branch']}_PC{number + 1}"
        for number in range(training_components.shape[1])
    ]

    train = train.drop(columns=sources)
    validation = validation.drop(columns=sources)
    train[component_names] = training_components
    validation[component_names] = validation_components
    return train, validation


## 8. Reproducibility and checkpoint identity

Hash all upstream inputs and screening definitions. A checkpoint is reused only when its configuration identity matches exactly.


In [15]:
RUN_VERSION = "final-notebook-05-v4-26-features-d28"


def file_digest(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


identity = {
    "run_version": RUN_VERSION,
    "input_sha256": {
        name: file_digest(path)
        for name, path in input_paths.items()
    },
    "selector_configurations": selector_manifest.to_dict("records"),
    "engineering_configurations": (
        engineering_manifest.where(pd.notna(engineering_manifest), None)
        .to_dict("records")
    ),
    "targets": ["direct", "stage_1", "stage_2"],
    "inner_folds": 5,
    "selection_metric": "mean inner-fold macro F1",
    "near_tie_margin": 0.01,
}
configuration_hash = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode()
).hexdigest()

run_manifest_path = output_directory / "run_manifest.json"
run_manifest = {
    **identity,
    "configuration_hash": configuration_hash,
    "python": platform.python_version(),
    "scikit_learn": sklearn_version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
if run_manifest_path.is_file():
    existing_manifest = json.loads(run_manifest_path.read_text())
    if existing_manifest.get("configuration_hash") != configuration_hash:
        raise RuntimeError(
            "Existing checkpoints belong to a different configuration. "
            "Preserve them and use a new output directory."
        )
else:
    run_manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n")

print("Configuration hash:", configuration_hash[:16])
print("Checkpoint identity is ready.")


Configuration hash: 1603aa55a898dcfc
Checkpoint identity is ready.


## 9. Work estimate

The two long-running cells checkpoint one split-seed/target unit at a time. Restarting the notebook skips every fully completed unit.


In [16]:
targets = ["direct", "stage_1", "stage_2"]
split_seeds = sorted(outer_assignments["split_seed"].unique())
selector_units = len(split_seeds) * len(targets)
engineering_units = len(split_seeds) * len(targets)
selector_fits = selector_units * 5 * len(selector_manifest)
engineering_fits = engineering_units * 5 * (1 + len(engineering_manifest))

work_plan = pd.DataFrame({
    "section": ["selector screen", "engineering screen", "total"],
    "resumable_units": [selector_units, engineering_units, selector_units + engineering_units],
    "reference_model_fits": [selector_fits, engineering_fits, selector_fits + engineering_fits],
    "checkpoint_boundary": [
        "one split seed × one target",
        "one split seed × one target",
        "two independent checkpoint files",
    ],
})
display(work_plan)
print("The notebook prints progress and a session-based ETA after every completed unit.")


,section,resumable_units,reference_model_fits,checkpoint_boundary
0,selector screen,60,2400,one split seed × one target
1,engineering screen,60,2400,one split seed × one target
2,total,120,4800,two independent checkpoint files


The notebook prints progress and a session-based ETA after every completed unit.


## 10. Run the checkpointed selector screen

This is the first long-running cell. It evaluates eight fixed selector/reference configurations across five inner folds for each seed and target.


In [17]:
def atomic_csv(frame, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


selector_checkpoint_path = output_directory / "selector_fold_checkpoint.csv"
selector_saved = (
    pd.read_csv(selector_checkpoint_path)
    if selector_checkpoint_path.is_file()
    else pd.DataFrame()
)
selector_expected_rows = 5 * len(selector_manifest)
selector_complete_units = set()
if not selector_saved.empty:
    saved_sizes = selector_saved.groupby(["split_seed", "target"]).size()
    selector_complete_units = set(
        saved_sizes[saved_sizes.eq(selector_expected_rows)].index
    )

selector_rows = (
    []
    if selector_saved.empty
    else selector_saved[
        selector_saved.apply(
            lambda row: (row["split_seed"], row["target"])
            in selector_complete_units,
            axis=1,
        )
    ].to_dict("records")
)

session_durations = []
print(
    f"Resuming with {len(selector_complete_units)}/{selector_units} "
    "selector units complete.",
    flush=True,
)

for split_seed in split_seeds:
    seed_inner = inner_assignments.loc[
        inner_assignments["split_seed"].eq(split_seed)
    ]
    outer_train_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )
    assert set(seed_inner["PATNO"]) == outer_train_ids
    assert outer_test_ids.isdisjoint(outer_train_ids)

    for target_name in targets:
        unit = (split_seed, target_name)
        if unit in selector_complete_units:
            continue

        unit_start = time.perf_counter()
        unit_rows = []
        print(
            f"Starting selector seed {split_seed:02d}, {target_name}: "
            f"{len(selector_manifest)} configurations × 5 folds",
            flush=True,
        )

        for inner_fold in range(5):
            validation_ids = sorted(
                seed_inner.loc[
                    seed_inner["inner_validation_fold"].eq(inner_fold),
                    "PATNO",
                ].tolist()
            )
            training_ids = sorted(outer_train_ids - set(validation_ids))
            assert set(training_ids).isdisjoint(validation_ids)
            assert outer_test_ids.isdisjoint([*training_ids, *validation_ids])

            train_ids, valid_ids, y_train, y_validation = target_data(
                target_name,
                training_ids,
                validation_ids,
            )
            random_seed = 100_000 + split_seed * 10 + inner_fold
            raw_train, X_train, X_validation, group_by_column = dense_pair(
                train_ids,
                valid_ids,
            )

            selected_cache = {}
            for configuration in selector_manifest.itertuples(index=False):
                cache_key = (configuration.selector, configuration.selector_parameter)
                if cache_key not in selected_cache:
                    selected_columns, selected_groups = selected_feature_columns(
                        configuration.selector,
                        configuration.selector_parameter,
                        raw_train,
                        X_train,
                        y_train,
                        group_by_column,
                        random_seed,
                    )
                    for group in selected_groups:
                        group_columns = {
                            column
                            for column, declared_group in group_by_column.items()
                            if declared_group == group
                        }
                        assert group_columns.issubset(selected_columns)
                    selected_cache[cache_key] = (selected_columns, selected_groups)

                selected_columns, selected_groups = selected_cache[cache_key]
                model = reference_model(configuration.reference_model, random_seed)
                fit_start = time.perf_counter()
                fit_checked(
                    model,
                    X_train[selected_columns],
                    y_train,
                    f"{configuration.reference_model} reference model",
                )
                predictions = model.predict(X_validation[selected_columns])

                unit_rows.append({
                    "split_seed": split_seed,
                    "target": target_name,
                    "inner_validation_fold": inner_fold,
                    "configuration_id": configuration.configuration_id,
                    "pipeline_key": configuration.pipeline_key,
                    "candidate_kind": configuration.candidate_kind,
                    "feature_set": configuration.feature_set,
                    "selector": configuration.selector,
                    "selector_parameter": configuration.selector_parameter,
                    "reference_model": configuration.reference_model,
                    "training_patients": len(y_train),
                    "validation_patients": len(y_validation),
                    "processed_input_columns": X_train.shape[1],
                    "selected_columns": len(selected_columns),
                    "selected_groups": len(selected_groups),
                    "selected_group_names": " | ".join(sorted(selected_groups)),
                    "selected_column_names": " | ".join(selected_columns),
                    "fit_seconds": time.perf_counter() - fit_start,
                    **classification_metrics(target_name, y_validation, predictions),
                })

        assert len(unit_rows) == selector_expected_rows
        selector_rows.extend(unit_rows)
        selector_complete_units.add(unit)
        selector_checkpoint = pd.DataFrame(selector_rows).sort_values(
            ["split_seed", "target", "inner_validation_fold", "configuration_id"]
        )
        atomic_csv(selector_checkpoint, selector_checkpoint_path)

        duration = time.perf_counter() - unit_start
        session_durations.append(duration)
        remaining = selector_units - len(selector_complete_units)
        eta_minutes = remaining * np.mean(session_durations) / 60
        print(
            f"Completed {len(selector_complete_units)}/{selector_units} selector units "
            f"in {duration / 60:.1f} min; estimated remaining {eta_minutes:.1f} min",
            flush=True,
        )

selector_folds = pd.DataFrame(selector_rows).sort_values(
    ["split_seed", "target", "inner_validation_fold", "configuration_id"]
).reset_index(drop=True)
print(f"Selector screen complete: {len(selector_folds):,} fold results.")


Resuming with 0/60 selector units complete.
Starting selector seed 00, direct: 8 configurations × 5 folds
Completed 1/60 selector units in 0.2 min; estimated remaining 14.6 min
Starting selector seed 00, stage_1: 8 configurations × 5 folds
Completed 2/60 selector units in 0.1 min; estimated remaining 10.8 min
Starting selector seed 00, stage_2: 8 configurations × 5 folds
Completed 3/60 selector units in 0.1 min; estimated remaining 9.1 min
Starting selector seed 01, direct: 8 configurations × 5 folds
Completed 4/60 selector units in 0.1 min; estimated remaining 8.6 min
Starting selector seed 01, stage_1: 8 configurations × 5 folds
Completed 5/60 selector units in 0.1 min; estimated remaining 7.7 min
Starting selector seed 01, stage_2: 8 configurations × 5 folds
Completed 6/60 selector units in 0.1 min; estimated remaining 7.0 min
Starting selector seed 02, direct: 8 configurations × 5 folds
Completed 7/60 selector units in 0.1 min; estimated remaining 6.6 min
Starting selector seed 02,

Summarize the five paired inner folds for every selector configuration within each split seed and target.


In [18]:
selector_summary = (
    selector_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "feature_set", "selector",
            "selector_parameter", "reference_model",
        ],
        as_index=False,
    )
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_priority_recall=("priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        mean_selected_groups=("selected_groups", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
)

selector_frequency_preview = (
    selector_summary.groupby(
        ["target", "selector", "selector_parameter", "reference_model"]
    )["mean_macro_f1"]
    .mean()
    .rename("descriptive_mean_macro_f1")
    .reset_index()
    .sort_values(["target", "descriptive_mean_macro_f1"], ascending=[True, False])
)
display(selector_frequency_preview.groupby("target").head(5))


,target,selector,selector_parameter,reference_model,descriptive_mean_macro_f1
0,direct,corrected_fdr,q<=0.05,extra_trees,0.506147
3,direct,extra_trees,threshold=median,extra_trees,0.503754
2,direct,extra_trees,threshold=1.25*median,extra_trees,0.502799
6,direct,none,all,extra_trees,0.496940
1,direct,corrected_fdr,q<=0.05,logistic,0.488165
14,stage_1,none,all,extra_trees,0.680621
8,stage_1,corrected_fdr,q<=0.05,extra_trees,0.678758
12,stage_1,l1,C=0.1,logistic,0.676332
10,stage_1,extra_trees,threshold=1.25*median,extra_trees,0.676243
11,stage_1,extra_trees,threshold=median,extra_trees,0.676041


## 11. Run the checkpointed engineering screen

This is the second long-running cell. Each engineered representation is compared with the unengineered 26-feature baseline using the same shallow Extra-Trees reference and identical folds.


In [19]:
engineering_checkpoint_path = output_directory / "engineering_fold_checkpoint.csv"
engineering_saved = (
    pd.read_csv(engineering_checkpoint_path)
    if engineering_checkpoint_path.is_file()
    else pd.DataFrame()
)
engineering_expected_rows = 5 * len(engineering_manifest)
engineering_complete_units = set()
if not engineering_saved.empty:
    saved_sizes = engineering_saved.groupby(["split_seed", "target"]).size()
    engineering_complete_units = set(
        saved_sizes[saved_sizes.eq(engineering_expected_rows)].index
    )

engineering_rows = (
    []
    if engineering_saved.empty
    else engineering_saved[
        engineering_saved.apply(
            lambda row: (row["split_seed"], row["target"])
            in engineering_complete_units,
            axis=1,
        )
    ].to_dict("records")
)

session_durations = []
print(
    f"Resuming with {len(engineering_complete_units)}/{engineering_units} "
    "engineering units complete.",
    flush=True,
)

for split_seed in split_seeds:
    seed_inner = inner_assignments.loc[
        inner_assignments["split_seed"].eq(split_seed)
    ]
    outer_train_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("train"),
            "PATNO",
        ]
    )
    outer_test_ids = set(
        outer_assignments.loc[
            outer_assignments["split_seed"].eq(split_seed)
            & outer_assignments["role"].eq("test"),
            "PATNO",
        ]
    )

    for target_name in targets:
        unit = (split_seed, target_name)
        if unit in engineering_complete_units:
            continue

        unit_start = time.perf_counter()
        unit_rows = []
        print(
            f"Starting engineering seed {split_seed:02d}, {target_name}: "
            f"{len(engineering_manifest)} branches × 5 folds",
            flush=True,
        )

        for inner_fold in range(5):
            validation_ids = sorted(
                seed_inner.loc[
                    seed_inner["inner_validation_fold"].eq(inner_fold),
                    "PATNO",
                ].tolist()
            )
            training_ids = sorted(outer_train_ids - set(validation_ids))
            assert set(training_ids).isdisjoint(validation_ids)
            assert outer_test_ids.isdisjoint([*training_ids, *validation_ids])

            train_ids, valid_ids, y_train, y_validation = target_data(
                target_name,
                training_ids,
                validation_ids,
            )
            random_seed = 200_000 + split_seed * 10 + inner_fold
            _, baseline_train, baseline_validation, _ = dense_pair(
                train_ids,
                valid_ids,
            )

            baseline_model = reference_model("extra_trees", random_seed)
            fit_checked(
                baseline_model,
                baseline_train,
                y_train,
                "Extra-Trees engineering baseline",
            )
            baseline_predictions = baseline_model.predict(baseline_validation)
            baseline_scores = classification_metrics(
                target_name,
                y_validation,
                baseline_predictions,
            )

            for branch_row in engineering_manifest.to_dict("records"):
                X_train, X_validation = engineered_pair(
                    baseline_train,
                    baseline_validation,
                    branch_row,
                )
                model = reference_model("extra_trees", random_seed)
                fit_start = time.perf_counter()
                fit_checked(
                    model,
                    X_train,
                    y_train,
                    f"Extra-Trees engineering branch {branch_row['branch']}",
                )
                predictions = model.predict(X_validation)
                candidate_scores = classification_metrics(
                    target_name,
                    y_validation,
                    predictions,
                )

                unit_rows.append({
                    "split_seed": split_seed,
                    "target": target_name,
                    "inner_validation_fold": inner_fold,
                    "configuration_id": branch_row["configuration_id"],
                    "pipeline_key": branch_row["pipeline_key"],
                    "candidate_kind": branch_row["candidate_kind"],
                    "feature_set": branch_row["feature_set"],
                    "selector": branch_row["selector"],
                    "selector_parameter": branch_row["selector_parameter"],
                    "branch": branch_row["branch"],
                    "branch_kind": branch_row["branch_kind"],
                    "reference_model": branch_row["reference_model"],
                    "training_patients": len(y_train),
                    "validation_patients": len(y_validation),
                    "selected_columns": X_train.shape[1],
                    "fit_seconds": time.perf_counter() - fit_start,
                    "baseline_macro_f1": baseline_scores["macro_f1"],
                    "baseline_priority_recall": baseline_scores["priority_recall"],
                    **candidate_scores,
                    "delta_macro_f1": (
                        candidate_scores["macro_f1"] - baseline_scores["macro_f1"]
                    ),
                    "delta_priority_recall": (
                        candidate_scores["priority_recall"]
                        - baseline_scores["priority_recall"]
                    ),
                })

        assert len(unit_rows) == engineering_expected_rows
        engineering_rows.extend(unit_rows)
        engineering_complete_units.add(unit)
        engineering_checkpoint = pd.DataFrame(engineering_rows).sort_values(
            ["split_seed", "target", "inner_validation_fold", "configuration_id"]
        )
        atomic_csv(engineering_checkpoint, engineering_checkpoint_path)

        duration = time.perf_counter() - unit_start
        session_durations.append(duration)
        remaining = engineering_units - len(engineering_complete_units)
        eta_minutes = remaining * np.mean(session_durations) / 60
        print(
            f"Completed {len(engineering_complete_units)}/{engineering_units} "
            f"engineering units in {duration / 60:.1f} min; "
            f"estimated remaining {eta_minutes:.1f} min",
            flush=True,
        )

engineering_folds = pd.DataFrame(engineering_rows).sort_values(
    ["split_seed", "target", "inner_validation_fold", "configuration_id"]
).reset_index(drop=True)
print(f"Engineering screen complete: {len(engineering_folds):,} fold results.")


Resuming with 0/60 engineering units complete.
Starting engineering seed 00, direct: 7 branches × 5 folds
Completed 1/60 engineering units in 0.1 min; estimated remaining 3.1 min
Starting engineering seed 00, stage_1: 7 branches × 5 folds
Completed 2/60 engineering units in 0.0 min; estimated remaining 2.9 min
Starting engineering seed 00, stage_2: 7 branches × 5 folds
Completed 3/60 engineering units in 0.0 min; estimated remaining 2.8 min
Starting engineering seed 01, direct: 7 branches × 5 folds
Completed 4/60 engineering units in 0.1 min; estimated remaining 2.8 min
Starting engineering seed 01, stage_1: 7 branches × 5 folds
Completed 5/60 engineering units in 0.0 min; estimated remaining 2.7 min
Starting engineering seed 01, stage_2: 7 branches × 5 folds
Completed 6/60 engineering units in 0.0 min; estimated remaining 2.6 min
Starting engineering seed 02, direct: 7 branches × 5 folds
Completed 7/60 engineering units in 0.1 min; estimated remaining 2.6 min
Starting engineering seed

Summarize paired engineering results. A branch is eligible only when mean macro-F1 improves, at least three folds improve, and the target-priority recall does not fall by more than 0.01.


In [20]:
engineering_summary = (
    engineering_folds.groupby(
        [
            "split_seed", "target", "configuration_id", "pipeline_key",
            "candidate_kind", "feature_set", "selector",
            "selector_parameter", "branch", "branch_kind", "reference_model",
        ],
        as_index=False,
    )
    .agg(
        mean_baseline_macro_f1=("baseline_macro_f1", "mean"),
        mean_macro_f1=("macro_f1", "mean"),
        sd_macro_f1=("macro_f1", "std"),
        mean_delta_macro_f1=("delta_macro_f1", "mean"),
        improving_folds=("delta_macro_f1", lambda values: int((values > 0).sum())),
        mean_priority_recall=("priority_recall", "mean"),
        mean_delta_priority_recall=("delta_priority_recall", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_selected_columns=("selected_columns", "mean"),
        inner_folds=("inner_validation_fold", "nunique"),
    )
)
engineering_summary["mean_selected_groups"] = engineering_summary[
    "mean_selected_columns"
]
engineering_summary["eligible_to_advance"] = (
    engineering_summary["mean_delta_macro_f1"].gt(0)
    & engineering_summary["improving_folds"].ge(3)
    & engineering_summary["mean_delta_priority_recall"].ge(-0.01)
)

engineering_frequency = (
    engineering_summary.loc[engineering_summary["eligible_to_advance"]]
    .groupby(["target", "branch", "branch_kind"])
    .size()
    .rename("eligible_split_seeds")
    .reset_index()
    .sort_values(["target", "eligible_split_seeds"], ascending=[True, False])
)
display(engineering_frequency.groupby("target").head(5))


,target,branch,branch_kind,eligible_split_seeds
1,direct,INT_GAIT_X_POSTURAL_STABILITY,interaction,9
2,direct,INT_MOBILITY_X_POSTURAL_STABILITY,interaction,8
5,direct,PCA_PART_I_PATIENT_3_90,pca,8
6,direct,PCA_PART_I_PATIENT_3_95,pca,8
0,direct,INT_DURATION_X_MOTOR,interaction,7
9,stage_1,INT_MOBILITY_X_POSTURAL_STABILITY,interaction,9
7,stage_1,INT_DURATION_X_MOTOR,interaction,7
8,stage_1,INT_GAIT_X_POSTURAL_STABILITY,interaction,7
10,stage_1,INT_MOTOR_X_COGNITION,interaction,7
11,stage_1,PCA_PART_I_PATIENT_3_80,pca,7


## 12. Retain two distinct feature pipelines

Combine selector configurations with locally eligible engineering branches. Apply the locked macro-F1 and target-priority-recall rule, then prevent duplicate raw pipelines evaluated by different reference models from occupying both slots.


In [21]:
selector_pool = selector_summary.copy()
selector_pool["branch"] = pd.NA
selector_pool["branch_kind"] = "raw"
selector_pool["mean_delta_macro_f1"] = np.nan
selector_pool["improving_folds"] = np.nan
selector_pool["mean_delta_priority_recall"] = np.nan
selector_pool["eligible_to_advance"] = True

engineering_pool = engineering_summary.copy()
pool_columns = [
    "split_seed", "target", "configuration_id", "pipeline_key",
    "candidate_kind", "feature_set", "selector", "selector_parameter",
    "branch", "branch_kind", "reference_model", "mean_macro_f1",
    "sd_macro_f1", "mean_priority_recall", "mean_balanced_accuracy",
    "mean_accuracy", "mean_selected_columns", "mean_selected_groups",
    "inner_folds", "mean_delta_macro_f1", "improving_folds",
    "mean_delta_priority_recall", "eligible_to_advance",
]
screening_pool = pd.concat(
    [selector_pool[pool_columns], engineering_pool[pool_columns]],
    ignore_index=True,
)

model_order = {"logistic": 0, "extra_trees": 1}


def retain_two_pipelines(group):
    remaining = group.loc[group["eligible_to_advance"]].copy()
    retained = []
    for rank in [1, 2]:
        if remaining.empty:
            raise RuntimeError("Fewer than two eligible feature pipelines remain.")
        best_macro_f1 = remaining["mean_macro_f1"].max()
        near_ties = remaining.loc[
            remaining["mean_macro_f1"].ge(best_macro_f1 - 0.01)
        ].copy()
        near_ties["reference_model_order"] = near_ties["reference_model"].map(
            model_order
        )
        chosen = near_ties.sort_values(
            [
                "mean_priority_recall", "mean_selected_groups",
                "mean_selected_columns", "reference_model_order",
                "pipeline_key", "configuration_id",
            ],
            ascending=[False, True, True, True, True, True],
        ).iloc[0]
        chosen_row = chosen.drop(labels="reference_model_order").to_dict()
        chosen_row["retained_rank"] = rank
        retained.append(chosen_row)
        remaining = remaining.loc[
            remaining["pipeline_key"].ne(chosen["pipeline_key"])
        ]
    return pd.DataFrame(retained)


retained_pipelines = pd.concat(
    [
        retain_two_pipelines(group)
        for _, group in screening_pool.groupby(
            ["split_seed", "target"],
            sort=False,
        )
    ],
    ignore_index=True,
).sort_values(["split_seed", "target", "retained_rank"])

retained_frequency = (
    retained_pipelines.groupby(
        ["target", "candidate_kind", "pipeline_key", "branch", "selector"],
        dropna=False,
    )
    .size()
    .rename("retained_count")
    .reset_index()
    .sort_values(["target", "retained_count"], ascending=[True, False])
)

display(retained_frequency.groupby("target").head(10))
print("Retained feature-pipeline rows:", len(retained_pipelines))


,target,candidate_kind,pipeline_key,branch,selector,retained_count
5,direct,selector_configuration,raw__corrected_fdr__q<=0.05,NaN,corrected_fdr,13
6,direct,selector_configuration,raw__extra_trees__threshold=1.25*median,NaN,extra_trees,7
7,direct,selector_configuration,raw__extra_trees__threshold=median,NaN,extra_trees,6
0,direct,engineered_representation,engineering__INT_DURATION_X_MOTOR,INT_DURATION_X_MOTOR,none,3
2,direct,engineered_representation,engineering__INT_MOTOR_X_COGNITION,INT_MOTOR_X_COGNITION,none,3
1,direct,engineered_representation,engineering__INT_GAIT_X_POSTURAL_STABILITY,INT_GAIT_X_POSTURAL_STABILITY,none,2
8,direct,selector_configuration,raw__l1__C=0.1,NaN,l1,2
9,direct,selector_configuration,raw__none__all,NaN,none,2
3,direct,engineered_representation,engineering__PCA_PART_I_PATIENT_3_80,PCA_PART_I_PATIENT_3_80,none,1
4,direct,engineered_representation,engineering__PCA_PART_I_PATIENT_3_90,PCA_PART_I_PATIENT_3_90,none,1


Retained feature-pipeline rows: 120


## 13. Validate the screening result

Confirm complete paired coverage, target-specific selection, valid metrics, distinct retained pipelines, and the absence of outer-test predictions.


In [22]:
validation_rows = []


def check(name, condition, detail):
    validation_rows.append({
        "check": name,
        "passed": bool(condition),
        "detail": detail,
    })


selector_unit_counts = selector_folds.groupby(["split_seed", "target"]).size()
engineering_unit_counts = engineering_folds.groupby(["split_seed", "target"]).size()
retained_counts = retained_pipelines.groupby(["split_seed", "target"]).size()
retained_distinct = retained_pipelines.groupby(
    ["split_seed", "target"]
)["pipeline_key"].nunique()
metric_columns = ["macro_f1", "priority_recall", "balanced_accuracy", "accuracy"]

check(
    "26-source candidate contract",
    len(features) == 26 and predictors.shape == (1040, 26),
    "one 1,040-patient final input",
)
check(
    "60 selector units complete",
    len(selector_unit_counts) == 60 and selector_unit_counts.eq(selector_expected_rows).all(),
    "20 seeds × 3 targets",
)
check(
    "five selector folds per configuration",
    selector_folds.groupby(
        ["split_seed", "target", "configuration_id"]
    )["inner_validation_fold"].nunique().eq(5).all(),
    "all eight configurations",
)
check(
    "four selector families screened",
    set(selector_folds["selector"]) == {"none", "corrected_fdr", "l1", "extra_trees"},
    "none, FDR, L1, and Extra Trees",
)
check(
    "every selected subset is nonempty",
    selector_folds["selected_columns"].gt(0).all(),
    "all selector fits",
)
check(
    "60 engineering units complete",
    len(engineering_unit_counts) == 60
    and engineering_unit_counts.eq(engineering_expected_rows).all(),
    "20 seeds × 3 targets",
)
check(
    "seven compatible engineering branches screened",
    engineering_folds["branch"].nunique() == 7,
    "four interactions plus three PCA-threshold branches",
)
check(
    "five paired folds per engineering branch",
    engineering_summary["inner_folds"].eq(5).all(),
    "same folds as the raw baseline",
)
check(
    "engineering sources remain inside final 26",
    all(
        set(text.split(" | ")).issubset(features)
        for text in engineering_manifest["source_features"]
    ),
    "no additional raw predictor",
)
check(
    "all screening metrics are bounded",
    selector_folds[metric_columns].apply(
        lambda column: column.between(0, 1).all()
    ).all()
    and engineering_folds[metric_columns].apply(
        lambda column: column.between(0, 1).all()
    ).all(),
    "macro F1, recalls, balanced accuracy, and accuracy",
)
check(
    "two pipelines retained per seed and target",
    len(retained_counts) == 60 and retained_counts.eq(2).all(),
    "120 queue rows",
)
check(
    "retained pipelines are distinct",
    retained_distinct.eq(2).all(),
    "reference-model duplicates cannot fill both slots",
)
check(
    "no outer-test result generated",
    not any(
        "outer_prediction" in column or "outer_metric" in column
        for column in [*selector_folds.columns, *engineering_folds.columns]
    ),
    "Notebook 05 uses inner validation only",
)
check(
    "no patient-level transformed matrix saved",
    True,
    "only configuration and fold summaries are written",
)
check(
    "checkpoint identity matches",
    json.loads(run_manifest_path.read_text())["configuration_hash"]
    == configuration_hash,
    "input and configuration hash",
)

validation = pd.DataFrame(validation_rows)
display(validation)
assert validation["passed"].all(), validation.loc[~validation["passed"]]
print(f"Validation checks passed: {validation['passed'].sum()}/{len(validation)}")


,check,passed,detail
0,26-source candidate contract,True,"one 1,040-patient final input"
1,60 selector units complete,True,20 seeds × 3 targets
2,five selector folds per configuration,True,all eight configurations
3,four selector families screened,True,"none, FDR, L1, and Extra Trees"
4,every selected subset is nonempty,True,all selector fits
5,60 engineering units complete,True,20 seeds × 3 targets
6,seven compatible engineering branches screened,True,four interactions plus three PCA-threshold bra...
7,five paired folds per engineering branch,True,same folds as the raw baseline
8,engineering sources remain inside final 26,True,no additional raw predictor
9,all screening metrics are bounded,True,"macro F1, recalls, balanced accuracy, and accu..."


Validation checks passed: 15/15


## 14. Save reusable screening artifacts

Write the completed manifests, fold results, summaries, retained queue, and validation table. Existing files must be numerically and textually equivalent; differing results are never overwritten silently.


In [23]:
def frames_equivalent(current, existing):
    if current.columns.tolist() != existing.columns.tolist() or current.shape != existing.shape:
        return False

    for column in current.columns:
        left = current[column]
        right = existing[column]
        left_numeric = pd.to_numeric(left, errors="coerce")
        right_numeric = pd.to_numeric(right, errors="coerce")
        left_numeric_ok = left_numeric.notna().eq(left.notna()).all()
        right_numeric_ok = right_numeric.notna().eq(right.notna()).all()

        if left_numeric_ok and right_numeric_ok:
            if not np.allclose(
                left_numeric.to_numpy(dtype=float),
                right_numeric.to_numpy(dtype=float),
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True,
            ):
                return False
        else:
            left_text = left.astype("string").fillna("<NA>").reset_index(drop=True)
            right_text = right.astype("string").fillna("<NA>").reset_index(drop=True)
            if not left_text.equals(right_text):
                return False
    return True


def save_new_or_equivalent(frame, path):
    if path.is_file():
        existing = pd.read_csv(path, low_memory=False)
        if not frames_equivalent(frame.reset_index(drop=True), existing):
            raise FileExistsError(
                f"Existing artifact truly differs and was not overwritten: {path.name}"
            )
        return "already equivalent"

    frame.to_csv(path, index=False)
    return "created"


artifacts = {
    "selector_configuration_manifest.csv": selector_manifest,
    "selector_inner_fold_results.csv": selector_folds,
    "selector_partition_summary.csv": selector_summary,
    "engineering_configuration_manifest.csv": engineering_manifest,
    "excluded_legacy_engineering_branches.csv": excluded_legacy_branches,
    "engineering_inner_fold_results.csv": engineering_folds,
    "engineering_partition_summary.csv": engineering_summary,
    "screening_candidate_pool.csv": screening_pool,
    "retained_feature_pipelines.csv": retained_pipelines,
    "retained_pipeline_frequency.csv": retained_frequency,
    "feature_pipeline_screening_validation.csv": validation,
}

save_rows = []
for filename, frame in artifacts.items():
    status = save_new_or_equivalent(frame, output_directory / filename)
    save_rows.append({"artifact": filename, "status": status, "rows": len(frame)})

save_summary = pd.DataFrame(save_rows)
display(save_summary)
print("Notebook 06 can start from retained_feature_pipelines.csv.")


,artifact,status,rows
0,selector_configuration_manifest.csv,created,8
1,selector_inner_fold_results.csv,created,2400
2,selector_partition_summary.csv,created,480
3,engineering_configuration_manifest.csv,created,7
4,excluded_legacy_engineering_branches.csv,created,6
5,engineering_inner_fold_results.csv,created,2100
6,engineering_partition_summary.csv,created,420
7,screening_candidate_pool.csv,created,900
8,retained_feature_pipelines.csv,created,120
9,retained_pipeline_frequency.csv,created,32


Notebook 06 can start from retained_feature_pipelines.csv.


## Interpretation boundary

The retained configurations are local to each split seed and target. Their pooled frequencies are descriptive and cannot define one global feature set.

Notebook 06 will refit each retained feature pipeline inside the same inner folds and compare all nine frozen model families. Outer test patients remain untouched until Notebook 07 completes focused tuning.
